# Prepare PARROT data for AAV capsid surface region variants

Prepare sequences and measured scores for AAV capsid surface region variants, split the data into training, validation, and test sets, and write the files used by PARROT.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.unicode_minus'] = False

## Data split

Split the data into training, validation, and test sets at a ratio of 0.8 : 0.1 : 0.1.

## Load and prepare the data

In [2]:
# Load the sequence, category, plasmid count, and viability score columns.
plm_data = "...\\pLM-bVAE\\reproduction_paper\\data\\"
aav_data = pd.read_csv(plm_data + "raw_data\\AAV_library_w_selection_scores.csv", sep=",", usecols=["aa_x", "category_x", "GAS1_plasmid_N", "GAS1_virus_S"]).reset_index(drop=True)

In [3]:
aav_data.shape

(243481, 4)

In [4]:
# Remove rows with negative infinite viability scores.
print("Rows with GAS1_virus_S equal to -inf:", (aav_data["GAS1_virus_S"] == -np.inf).sum())
aav_data = aav_data[aav_data["GAS1_virus_S"] != -np.inf].reset_index(drop=True)

# Select variants with plasmid counts greater than 100.
print("Rows with GAS1_plasmid_N greater than 100:", (aav_data["GAS1_plasmid_N"] > 100).sum())
aav_data_filtered = aav_data[aav_data["GAS1_plasmid_N"] > 100].reset_index(drop=True)

# Convert the sequences to uppercase.
aav_data_filtered["aa_x"] = aav_data_filtered["aa_x"].str.upper()

Rows with GAS1_virus_S equal to -inf: 17840
Rows with GAS1_plasmid_N greater than 100: 173979


In [5]:
aav_data_filtered.shape

(173979, 4)

In [6]:
# For duplicate sequences, retain the row with the highest viability score.
print("Rows with duplicate aa_x sequences:", aav_data_filtered["aa_x"].duplicated().sum())
aav_data_filtered_deldp = aav_data_filtered.sort_values("GAS1_virus_S", ascending=False).drop_duplicates(subset="aa_x", keep="first").reset_index(drop=True)

Rows with duplicate aa_x sequences: 9383


In [7]:
aav_data_filtered_deldp.shape

(164596, 4)

In [8]:
# Count rows in categories beginning with rnn.
print("Rows with category_x starting with rnn:", aav_data_filtered_deldp["category_x"].str.startswith("rnn").sum())

# Count rows in categories beginning with cnn.
print("Rows with category_x starting with cnn:", aav_data_filtered_deldp["category_x"].str.startswith("cnn").sum())

Rows with category_x starting with rnn: 48113
Rows with category_x starting with cnn: 47805


In [9]:
# Count rows in the rnn_designed_plus_rand_train_walked category.
print("Rows with category_x equal to rnn_designed_plus_rand_train_walked:", (aav_data_filtered_deldp["category_x"] == "rnn_designed_plus_rand_train_walked").sum())

Rows with category_x equal to rnn_designed_plus_rand_train_walked: 16742


In [10]:
# Select the rnn_designed_plus_rand_train_walked category.
aav_data_filtered_deldp_categorized = aav_data_filtered_deldp[aav_data_filtered_deldp["category_x"] == "rnn_designed_plus_rand_train_walked"]
aav_data_filtered_deldp_categorized.shape

(16742, 4)

In [11]:
# Inspect the minimum and maximum sequence lengths.
print("Maximum aa_x sequence length:", aav_data_filtered_deldp_categorized["aa_x"].str.len().max())
print("Minimum aa_x sequence length:", aav_data_filtered_deldp_categorized["aa_x"].str.len().min())

Maximum aa_x sequence length: 43
Minimum aa_x sequence length: 28


In [12]:
# Prepend the wild-type sequence if it is present in the filtered data.
wt_seq = "DEEEIRTTNPVATEQYGSVSTNLQRGNR"
wt_row = aav_data_filtered_deldp[aav_data_filtered_deldp["aa_x"] == wt_seq]

aav_data_filtered_deldp_categorized_wt = pd.concat([wt_row, aav_data_filtered_deldp_categorized], ignore_index=True)
aav_data_filtered_deldp_categorized_wt.head(1)

,aa_x,category_x,GAS1_plasmid_N,GAS1_virus_S
0,DEEEIRTTNPVATEQYGSVSTNLQRGNR,wild_type,192,1.070797


In [13]:
# Drop the category and plasmid count columns.
aav_data_region = aav_data_filtered_deldp_categorized_wt.drop(columns=["category_x", "GAS1_plasmid_N"])

# Rename the sequence and viability score columns.
aav_data_region = aav_data_region.rename(columns={"aa_x": "seq", "GAS1_virus_S": "score"})

In [14]:
aav_data_region

,seq,score
0,DEEEIRTTNPVATEQYGSVSTNLQRGNR,1.070797
1,DETEIATTNPVATEQYGDVSTNLQGGGNER,3.734888
2,DEAEIRTTNPVATEQYGSVSNNLQKGGNDR,3.727338
3,DEAEIRTTNPVATEQYGVVSTNLQGGMQ,3.449196
4,NEEEISTTNPVATEQYGSVSTNLQHNGNP,3.250796
...,...,...
16738,DESEISENATTNPVAYECWGSVYYIENGGQQGQNPRH,-9.875272
16739,ATECWGIVATTIGINGGNHNGQPQTPLQEGEGHNDS,-9.979250
16740,DEATENTEIATNPVAYEQWGTVCPTDNGDQHQEDDHY,-10.035466
16741,ATECWGIVATTIGINGGNHNGQPVTPHQRGEGHNQS,-10.071762


## Random data split

### PARROT split file

In [15]:
# Shuffle the AAV data.
aav_data_region_shuffled = aav_data_region.sample(frac=1, random_state=42).reset_index(drop=True)

# Split the shuffled data into training, validation, and test sets at a ratio of 0.8:0.1:0.1.
train_size = int(0.8 * len(aav_data_region_shuffled))
dev_size = int(0.1 * len(aav_data_region_shuffled))
aav_train = aav_data_region_shuffled[:train_size]
aav_dev = aav_data_region_shuffled[train_size:train_size + dev_size]
aav_test = aav_data_region_shuffled[train_size + dev_size:]

In [16]:
aav_data_region_shuffled

,seq,score
0,DEAEICTTNPVAYEQYGTVSTTNGQMNDDER,0.487101
1,DEHEIRTTNPVATEQYGEVSTNLQHQGNEP,1.515530
2,DEDEIATTNPVATECYGSVCQTLDQMQREGGDQGD,1.038128
3,DEEEIATTNPVATEQYGSVSTNLQGAGGAN,2.331957
4,TNSSEISEATTNPVAYEQWGEVYTLGGLGNEGQNDTD,-5.847045
...,...,...
16738,NTEAEIATTNPVAYEAWGNICATIDNQGEDSGNMDR,-3.583304
16739,DETDEIATTNPVAYEQWGCVSGGNGLLQDHPNGGS,-4.239651
16740,DEAEIATTNPVATEQYGIVSTNLQAGGNQ,0.845638
16741,DEHEIRTTNPVATEQYGSVSTTGPEQGLGNMGQ,1.629094


In [ ]:
# Save the shuffled AAV data.
aav_data_region_shuffled.to_csv(plm_data + "parrot\\aav\\aav_random.txt", sep=" ", columns=["seq", "score"], header=False)

## Prepare PARROT optimization inputs

In [17]:
# Combine the training and validation data.
aav_train_dev = pd.concat([aav_train, aav_dev], ignore_index=True)
aav_train_dev

,seq,score
0,DEAEICTTNPVAYEQYGTVSTTNGQMNDDER,0.487101
1,DEHEIRTTNPVATEQYGEVSTNLQHQGNEP,1.515530
2,DEDEIATTNPVATECYGSVCQTLDQMQREGGDQGD,1.038128
3,DEEEIATTNPVATEQYGSVSTNLQGAGGAN,2.331957
4,TNSSEISEATTNPVAYEQWGEVYTLGGLGNEGQNDTD,-5.847045
...,...,...
15063,NQEEAEIATTNPVATEYWGVVVHTENHDQDEQGGNQD,-5.167110
15064,SETETEIATTNPVAYEPWGAVCIENHMQNNQHGNHN,-6.355555
15065,DETEIATTNPVAYEPWGSVYPNDDQDQDTGRNDKS,-0.237834
15066,DEATEENTEIATNPVAYEQWGTVCPTDNYGDQHQGDQH,-6.704319


In [18]:
# Check for duplicate sequences within each split.
print(aav_train_dev.shape[0], aav_test.shape[0])

# Check for sequences shared between splits.
print(set(aav_train_dev["seq"]) & set(aav_test["seq"]))

# Check the split sizes and proportions.
aav_train_dev.shape[0] / aav_data_region_shuffled.shape[0], aav_test.shape[0] / aav_data_region_shuffled.shape[0]

15068 1675
set()


(0.8999581914830078, 0.10004180851699218)

In [ ]:
# Save the combined training and validation data for PARROT optimization.
aav_train_dev.to_csv(plm_data + "parrot\\aav\\aav_train_dev.txt", sep=" ", columns=["seq", "score"], header=False)

## Save the split indices

In [19]:
# Collect the training, validation, and test indices in separate rows.
index_df = pd.DataFrame({
    "train": " ".join(map(str, aav_train.index)),
    "dev": " ".join(map(str, aav_dev.index)),
    "test": " ".join(map(str, aav_test.index))
}, index=[0])

# Transpose the index table.
index_df = index_df.T.reset_index()
index_df.columns = ["split", "indices"]

In [20]:
index_df

,split,indices
0,train,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18...
1,dev,13394 13395 13396 13397 13398 13399 13400 1340...
2,test,15068 15069 15070 15071 15072 15073 15074 1507...


In [ ]:
# Write the split indices as unquoted text.
with open(plm_data + r"parrot\aav\aav_random_indices.txt", "w", encoding="utf-8") as f:
    for s in index_df["indices"]:
        f.write(str(s) + "\n")